In [1]:
%%writefile requirements.txt
streamlit>=1.35
plotly>=5.20
pandas>=2.0
PyPDF2>=3.0
docx2txt>=0.8

Overwriting requirements.txt


In [3]:
!pip install -r requirements.txt

In [5]:
%%writefile app.py
"""
CareerOS — The AI Career Operating System for Freshers
Quackathon 2026 — Track 3: "I Can Do It Better"

Single-file Streamlit MVP covering the core loop:
Landing -> Resume Upload -> ATS Analyzer -> Career Health Dashboard
-> Skill Gap Intelligence -> Career Twin
"""

import io
import re
import random
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

# Optional parsers — app degrades gracefully if not installed
try:
    import PyPDF2
    HAS_PDF = True
except ImportError:
    HAS_PDF = False

try:
    import docx2txt
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False


# ============================================================
# CONFIG & ROLE SKILL PROFILES
# ============================================================

st.set_page_config(
    page_title="CareerOS — AI Career Operating System",
    page_icon="🧭",
    layout="wide",
    initial_sidebar_state="expanded",
)

ROLE_PROFILES = {
    "Data Analyst": {
        "skills": ["SQL", "Excel", "Python", "Power BI", "DAX", "Power Query",
                   "Statistics", "A/B Testing", "Tableau", "ETL", "Pandas",
                   "NumPy", "Data Visualization", "Data Cleaning", "Storytelling"],
        "priority_weight": {"DAX": 5, "Statistics": 5, "A/B Testing": 4, "Power Query": 4,
                             "Tableau": 3, "ETL": 3, "SQL": 5, "Python": 4},
    },
    "Data Scientist": {
        "skills": ["Python", "SQL", "Machine Learning", "Scikit-learn", "Statistics",
                   "Pandas", "NumPy", "Deep Learning", "A/B Testing", "Feature Engineering",
                   "Model Deployment", "Data Visualization", "Excel", "Power BI"],
        "priority_weight": {"Machine Learning": 5, "Statistics": 5, "Feature Engineering": 4,
                             "Model Deployment": 4, "Deep Learning": 3},
    },
    "Business Analyst": {
        "skills": ["Excel", "SQL", "Power BI", "Stakeholder Management", "Requirements Gathering",
                   "Process Mapping", "Statistics", "Tableau", "Communication", "Data Visualization"],
        "priority_weight": {"Stakeholder Management": 5, "Requirements Gathering": 5,
                             "Process Mapping": 4, "SQL": 4},
    },
    "BI Developer": {
        "skills": ["SQL", "Power BI", "DAX", "Power Query", "ETL", "Data Warehousing",
                   "Tableau", "Python", "Data Modeling", "Excel"],
        "priority_weight": {"DAX": 5, "Data Warehousing": 5, "Data Modeling": 4, "ETL": 4},
    },
}

DEMO_RESUME = """
Neelima Perla
Data Analyst | Visakhapatnam, India

EDUCATION
BSc Data Science, Aditya Degree College — GPA 9.0

SKILLS
SQL, Python, Pandas, NumPy, Scikit-learn, Power BI, Tableau, ETL, EDA, Machine Learning,
Random Forest, SVM, Excel, Power Query

EXPERIENCE
Data Analysis Intern — Labmentix
Built ETL pipelines and performed SQL-based data cleaning at scale, processing 300K+ records
at 99% accuracy.

Power BI Intern — Ulearn
Built KPI dashboards using DAX and Power Query, reducing reporting effort by 15-30% and
improving refresh performance by 30%.

PROJECTS
EmotionOS — ML-based burnout and emotion classification system using Random Forest and SVM,
achieving 88-89% accuracy.

Bird Species Observation Analysis — exploratory data analysis and classification project.
"""


# ============================================================
# SESSION STATE
# ============================================================

defaults = {
    "page": "Landing",
    "resume_text": "",
    "candidate_name": "Candidate",
    "target_role": "Data Analyst",
    "analyzed": False,
    "ats_score": None,
    "matched_keywords": [],
    "missing_keywords": [],
    "formatting_issues": [],
    "health_scores": {},
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v


# ============================================================
# STYLING — modern SaaS / glassmorphism
# ============================================================

st.markdown("""
<style>
    .stApp { background: linear-gradient(135deg, #0B1228 0%, #122047 45%, #1A2B4C 100%); }
    section[data-testid="stSidebar"] { background: #0B1228; border-right: 1px solid rgba(255,255,255,0.08); }
    h1, h2, h3, h4, p, span, label, div { color: #EAF1F8; }
    .glass-card {
        background: rgba(255,255,255,0.06);
        backdrop-filter: blur(14px);
        border: 1px solid rgba(255,255,255,0.12);
        border-radius: 16px;
        padding: 28px;
        margin-bottom: 18px;
    }
    .metric-pill {
        display:inline-block; padding: 4px 14px; border-radius: 999px;
        background: rgba(46,117,182,0.25); border: 1px solid #2E75B6;
        font-size: 13px; color: #9FC6EA; margin-right: 8px;
    }
    .hero-title { font-size: 56px; font-weight: 800; line-height: 1.1;
        background: linear-gradient(90deg, #FFFFFF, #9FC6EA);
        -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    .hero-sub { font-size: 19px; color: #B7C4DA; margin-top: 6px; }
    .badge-ok { color: #4ADE80; font-weight: 600; }
    .badge-missing { color: #F87171; font-weight: 600; }
    .twin-bubble {
        background: linear-gradient(135deg, rgba(46,117,182,0.25), rgba(154,82,255,0.15));
        border: 1px solid rgba(154,82,255,0.4); border-radius: 18px; padding: 26px;
    }
    div[data-testid="stMetricValue"] { color: #EAF1F8; }
    .stButton>button {
        background: linear-gradient(90deg, #2E75B6, #5B9FE0); color: white; border: none;
        border-radius: 10px; padding: 10px 22px; font-weight: 600;
    }
    .stProgress > div > div > div > div { background: linear-gradient(90deg, #2E75B6, #5B9FE0); }
</style>
""", unsafe_allow_html=True)


# ============================================================
# CORE LOGIC — resume parsing & scoring (heuristic / explainable)
# ============================================================

def extract_text(uploaded_file):
    name = uploaded_file.name.lower()
    if name.endswith(".pdf") and HAS_PDF:
        reader = PyPDF2.PdfReader(uploaded_file)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if name.endswith(".docx") and HAS_DOCX:
        return docx2txt.process(uploaded_file)
    if name.endswith(".txt"):
        return uploaded_file.read().decode("utf-8", errors="ignore")
    return ""


def analyze_resume(text, role):
    profile = ROLE_PROFILES[role]
    text_lower = text.lower()

    matched = [s for s in profile["skills"] if s.lower() in text_lower]
    missing = [s for s in profile["skills"] if s.lower() not in text_lower]
    keyword_match_pct = len(matched) / max(len(profile["skills"]), 1)

    # Formatting heuristics
    issues = []
    if len(text.strip()) < 400:
        issues.append("Resume content is too short — ATS systems may flag thin sections")
    if not re.search(r"experience|intern", text_lower):
        issues.append("No clearly labelled Experience/Internship section detected")
    if not re.search(r"project", text_lower):
        issues.append("No clearly labelled Projects section detected")
    if not re.search(r"\d+%|\d+\+", text):
        issues.append("Few quantified achievements (numbers, %, +) found — weakens impact")
    if len(re.findall(r"\n", text)) < 5:
        issues.append("Formatting looks unstructured — check section breaks for ATS parsers")

    formatting_score = max(0, 100 - len(issues) * 12)
    ats_score = round(keyword_match_pct * 70 + (formatting_score / 100) * 30)
    ats_score = max(5, min(98, ats_score))

    # Career Health sub-scores (explainable, derived from same signals)
    resume_quality = round(min(100, formatting_score * 0.6 + (40 if len(text) > 800 else 20)))
    skill_relevance = round(keyword_match_pct * 100)
    has_quant = bool(re.search(r"\d+%|\d+\+|\d+k|\d{3,}", text_lower))
    project_strength = round(min(100, (60 if "project" in text_lower else 20) + (30 if has_quant else 0) + 10))
    market_readiness = round((skill_relevance * 0.6) + (resume_quality * 0.2) + (project_strength * 0.2))

    health = {
        "Resume Quality": resume_quality,
        "Skill Relevance": skill_relevance,
        "Project Strength": min(project_strength, 100),
        "Market Readiness": min(market_readiness, 100),
    }

    # Priority order for missing skills
    weights = profile["priority_weight"]
    missing_ranked = sorted(missing, key=lambda s: -weights.get(s, 1))

    return {
        "ats_score": ats_score,
        "matched": matched,
        "missing": missing,
        "missing_ranked": missing_ranked,
        "issues": issues,
        "health": health,
    }


def generate_twin_recommendations(result, role, name):
    missing = result["missing_ranked"][:3]
    health = result["health"]
    overall = round(sum(health.values()) / len(health))

    recs = []
    if missing:
        recs.append(f"Learn **{missing[0]}** — it's the highest-leverage gap for {role} roles right now.")
    if len(missing) > 1:
        recs.append(f"Add a project or resume line that demonstrates **{missing[1]}** with a measurable outcome.")
    if health["Project Strength"] < 80:
        recs.append("Quantify at least one existing project with a number — accuracy, % improvement, or scale.")
    if len(recs) < 3:
        recs.append(f"Apply to 15–20 fresh {role} postings this week to keep your funnel active while you upskill.")
    recs = recs[:3]

    projected = min(100, overall + random.randint(7, 12))
    return recs, overall, projected


# ============================================================
# SIDEBAR NAVIGATION
# ============================================================

with st.sidebar:
    st.markdown("## 🧭 CareerOS")
    st.caption("AI Career Operating System")
    st.markdown("---")
    pages = ["Landing", "Resume Upload", "ATS Analyzer", "Career Health Dashboard",
             "Skill Gap Analysis", "Career Twin"]
    icons = ["🏠", "📤", "🔍", "📊", "🧩", "🤖"]
    for pg, ic in zip(pages, icons):
        locked = pg != "Landing" and pg != "Resume Upload" and not st.session_state.analyzed
        label = f"{ic}  {pg}" + ("  🔒" if locked else "")
        if st.button(label, key=f"nav_{pg}", use_container_width=True, disabled=locked):
            st.session_state.page = pg
    st.markdown("---")
    if st.session_state.analyzed:
        st.success(f"Analyzed for: {st.session_state.target_role}")
    else:
        st.info("Upload a resume to unlock the full flow")


# ============================================================
# PAGE 1 — LANDING
# ============================================================

def page_landing():
    st.markdown('<div class="hero-title">CareerOS</div>', unsafe_allow_html=True)
    st.markdown('<div class="hero-sub">The AI Career Operating System for Freshers — '
                'don\'t just find jobs, know exactly why you\'re not getting hired.</div>',
                unsafe_allow_html=True)
    st.write("")
    c1, c2, c3 = st.columns(3)
    for col, title, desc, icon in zip(
        [c1, c2, c3],
        ["ATS Analyzer", "Skill Gap Engine", "Career Twin"],
        ["Know your ATS score before you apply — not after you get rejected.",
         "See exactly which skills are missing for your target role, ranked by impact.",
         "An AI coach that tells you the 3 highest-impact things to do this week."],
        ["🔍", "🧩", "🤖"]
    ):
        with col:
            st.markdown(f"""<div class="glass-card">
                <div style="font-size:32px">{icon}</div>
                <h3>{title}</h3>
                <p style="color:#B7C4DA">{desc}</p>
                </div>""", unsafe_allow_html=True)

    st.markdown("### Why CareerOS")
    b1, b2 = st.columns(2)
    with b1:
        st.markdown("""<div class="glass-card">
        <span class="metric-pill">For Freshers</span><br><br>
        Most graduates apply to 50+ jobs with zero feedback. CareerOS replaces silence with
        a clear score, a clear gap, and a clear next step.
        </div>""", unsafe_allow_html=True)
    with b2:
        st.markdown("""<div class="glass-card">
        <span class="metric-pill">Beyond LinkedIn</span><br><br>
        LinkedIn shows you jobs. CareerOS shows you why you're not getting them — and what
        to fix this week.
        </div>""", unsafe_allow_html=True)

    st.write("")
    if st.button("🚀 Upload Resume & Get Started", use_container_width=False):
        st.session_state.page = "Resume Upload"
        st.rerun()


# ============================================================
# PAGE 2 — RESUME UPLOAD
# ============================================================

def page_upload():
    st.markdown("## 📤 Upload Resume")
    st.markdown('<div class="glass-card">', unsafe_allow_html=True)

    name = st.text_input("Your name", value=st.session_state.candidate_name)
    role = st.selectbox("Target Role", list(ROLE_PROFILES.keys()),
                         index=list(ROLE_PROFILES.keys()).index(st.session_state.target_role))
    uploaded = st.file_uploader("Drag & drop your resume (PDF / DOCX / TXT)",
                                 type=["pdf", "docx", "txt"])

    colA, colB = st.columns(2)
    with colA:
        analyze_clicked = st.button("Analyze Resume", use_container_width=True)
    with colB:
        demo_clicked = st.button("Use Demo Resume Instead", use_container_width=True)

    st.markdown('</div>', unsafe_allow_html=True)

    if analyze_clicked:
        if uploaded is None:
            st.warning("Upload a file first, or click 'Use Demo Resume Instead'.")
        else:
            text = extract_text(uploaded)
            if not text.strip():
                st.error("Couldn't extract text from that file — try the demo resume.")
            else:
                _run_analysis(text, role, name)

    if demo_clicked:
        _run_analysis(DEMO_RESUME, role, name)


def _run_analysis(text, role, name):
    st.session_state.resume_text = text
    st.session_state.candidate_name = name or "Candidate"
    st.session_state.target_role = role
    result = analyze_resume(text, role)
    st.session_state.ats_score = result["ats_score"]
    st.session_state.matched_keywords = result["matched"]
    st.session_state.missing_keywords = result["missing_ranked"]
    st.session_state.formatting_issues = result["issues"]
    st.session_state.health_scores = result["health"]
    st.session_state.analyzed = True
    st.session_state.page = "ATS Analyzer"
    st.rerun()


# ============================================================
# PAGE 3 — ATS ANALYZER
# ============================================================

def page_ats():
    st.markdown("## 🔍 ATS Analyzer")
    score = st.session_state.ats_score

    c1, c2 = st.columns([1, 1.4])
    with c1:
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=score,
            number={'suffix': "%", 'font': {'size': 44, 'color': '#EAF1F8'}},
            gauge={
                'axis': {'range': [0, 100], 'tickcolor': '#EAF1F8'},
                'bar': {'color': "#2E75B6"},
                'bgcolor': "rgba(0,0,0,0)",
                'steps': [
                    {'range': [0, 50], 'color': 'rgba(248,113,113,0.25)'},
                    {'range': [50, 75], 'color': 'rgba(251,191,36,0.25)'},
                    {'range': [75, 100], 'color': 'rgba(74,222,128,0.25)'},
                ],
            },
            title={'text': "ATS Score", 'font': {'color': '#EAF1F8', 'size': 18}}
        ))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", height=320, margin=dict(t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        st.markdown('<div class="glass-card">', unsafe_allow_html=True)
        st.markdown(f"**Target role:** {st.session_state.target_role}")
        st.markdown(f"**Matched keywords:** {len(st.session_state.matched_keywords)}")
        st.markdown(f"**Missing keywords:** {len(st.session_state.missing_keywords)}")
        st.markdown(f"**Formatting issues:** {len(st.session_state.formatting_issues)}")
        st.markdown(f"**Recommendations:** {min(3, len(st.session_state.missing_keywords) + len(st.session_state.formatting_issues))}")
        st.markdown('</div>', unsafe_allow_html=True)

    cm, cn = st.columns(2)
    with cm:
        st.markdown("#### ✅ Matched Keywords")
        for k in st.session_state.matched_keywords:
            st.markdown(f'<span class="badge-ok">✓ {k}</span>', unsafe_allow_html=True)
    with cn:
        st.markdown("#### ❌ Missing Keywords")
        for k in st.session_state.missing_keywords[:8]:
            st.markdown(f'<span class="badge-missing">✗ {k}</span>', unsafe_allow_html=True)

    st.markdown("#### ⚠️ Formatting Issues")
    if st.session_state.formatting_issues:
        for i in st.session_state.formatting_issues:
            st.warning(i)
    else:
        st.success("No major formatting issues detected.")

    if st.button("Continue to Career Health Dashboard →"):
        st.session_state.page = "Career Health Dashboard"
        st.rerun()


# ============================================================
# PAGE 4 — CAREER HEALTH DASHBOARD
# ============================================================

def page_health():
    st.markdown("## 📊 Career Health Dashboard")
    health = st.session_state.health_scores
    overall = round(sum(health.values()) / len(health)) if health else 0

    c1, c2 = st.columns([1, 1.6])
    with c1:
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=overall,
            number={'suffix': " / 100", 'font': {'size': 38, 'color': '#EAF1F8'}},
            gauge={'axis': {'range': [0, 100]}, 'bar': {'color': "#9A52FF"},
                   'bgcolor': "rgba(0,0,0,0)"},
            title={'text': "Career Health Score", 'font': {'color': '#EAF1F8', 'size': 18}}
        ))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", height=300, margin=dict(t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        df = pd.DataFrame({"Category": list(health.keys()), "Score": list(health.values())})
        fig2 = px.bar(df, x="Score", y="Category", orientation="h", range_x=[0, 100],
                       color="Score", color_continuous_scale=["#F87171", "#FBBF24", "#4ADE80"])
        fig2.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                            font_color="#EAF1F8", height=300, margin=dict(t=10, b=10),
                            coloraxis_showscale=False)
        st.plotly_chart(fig2, use_container_width=True)

    cols = st.columns(4)
    for col, (label, val) in zip(cols, health.items()):
        with col:
            st.metric(label, f"{val}")

    fig3 = go.Figure(go.Scatterpolar(
        r=list(health.values()) + [list(health.values())[0]],
        theta=list(health.keys()) + [list(health.keys())[0]],
        fill='toself', line_color="#2E75B6"
    ))
    fig3.update_layout(polar=dict(bgcolor="rgba(0,0,0,0)",
                        radialaxis=dict(visible=True, range=[0, 100], color="#B7C4DA"),
                        angularaxis=dict(color="#EAF1F8")),
                        paper_bgcolor="rgba(0,0,0,0)", font_color="#EAF1F8",
                        showlegend=False, height=380, margin=dict(t=20, b=20))
    st.plotly_chart(fig3, use_container_width=True)

    if st.button("Continue to Skill Gap Analysis →"):
        st.session_state.page = "Skill Gap Analysis"
        st.rerun()


# ============================================================
# PAGE 5 — SKILL GAP ANALYSIS
# ============================================================

def page_skills():
    st.markdown("## 🧩 Skill Gap Intelligence")
    st.caption(f"Benchmarked against: {st.session_state.target_role}")

    c1, c2 = st.columns(2)
    with c1:
        st.markdown('<div class="glass-card">', unsafe_allow_html=True)
        st.markdown("#### Current Skills")
        for s in st.session_state.matched_keywords:
            st.markdown(f'<span class="badge-ok">✓ {s}</span><br>', unsafe_allow_html=True)
        st.markdown('</div>', unsafe_allow_html=True)
    with c2:
        st.markdown('<div class="glass-card">', unsafe_allow_html=True)
        st.markdown("#### Missing Skills")
        for s in st.session_state.missing_keywords:
            st.markdown(f'<span class="badge-missing">✗ {s}</span><br>', unsafe_allow_html=True)
        st.markdown('</div>', unsafe_allow_html=True)

    st.markdown("#### Priority Learning Order")
    for i, s in enumerate(st.session_state.missing_keywords[:5], start=1):
        st.markdown(f"**{i}. {s}**")
        st.progress(max(10, 100 - i * 15))

    total = len(st.session_state.matched_keywords) + len(st.session_state.missing_keywords)
    have = len(st.session_state.matched_keywords)
    fig = go.Figure(data=[go.Pie(
        labels=["Skills you have", "Skills to learn"],
        values=[have, total - have], hole=0.6,
        marker=dict(colors=["#4ADE80", "#F87171"])
    )])
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font_color="#EAF1F8", height=320,
                       margin=dict(t=10, b=10))
    st.plotly_chart(fig, use_container_width=True)

    if st.button("Continue to Career Twin →"):
        st.session_state.page = "Career Twin"
        st.rerun()


# ============================================================
# PAGE 6 — CAREER TWIN
# ============================================================

def page_twin():
    st.markdown("## 🤖 Career Twin")
    st.caption("Your AI version — continuously evaluating skills, projects, resume, and applications")

    result = {
        "missing_ranked": st.session_state.missing_keywords,
        "health": st.session_state.health_scores,
    }
    recs, overall, projected = generate_twin_recommendations(
        result, st.session_state.target_role, st.session_state.candidate_name
    )

    st.markdown(f"""<div class="twin-bubble">
        <h3>Hello {st.session_state.candidate_name}.</h3>
        <p style="font-size:17px; color:#D8E2F0">If I were you, here's what I'd do this week:</p>
        <ol style="font-size:16px; line-height:2;">
            {''.join(f'<li>{r}</li>' for r in recs)}
        </ol>
        </div>""", unsafe_allow_html=True)

    st.write("")
    c1, c2, c3 = st.columns(3)
    c1.metric("Current Career Health Score", f"{overall}")
    c2.metric("Projected After This Week", f"{projected}", delta=f"+{projected-overall}")
    c3.metric("Target Role", st.session_state.target_role)

    df = pd.DataFrame({
        "Stage": ["This week", "Projected"],
        "Score": [overall, projected]
    })
    fig = px.bar(df, x="Stage", y="Score", range_y=[0, 100], color="Stage",
                 color_discrete_sequence=["#2E75B6", "#9A52FF"])
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                       font_color="#EAF1F8", showlegend=False, height=320, margin=dict(t=10, b=10))
    st.plotly_chart(fig, use_container_width=True)

    st.info("Career Twin re-evaluates every time you update your resume or skills — "
            "this is a living recommendation, not a one-time report.")

    if st.button("🔄 Start Over with a New Resume"):
        for k, v in defaults.items():
            st.session_state[k] = v
        st.rerun()


# ============================================================
# ROUTER
# ============================================================

PAGE_FUNCS = {
    "Landing": page_landing,
    "Resume Upload": page_upload,
    "ATS Analyzer": page_ats,
    "Career Health Dashboard": page_health,
    "Skill Gap Analysis": page_skills,
    "Career Twin": page_twin,
}

if st.session_state.page in ("ATS Analyzer", "Career Health Dashboard", "Skill Gap Analysis", "Career Twin") \
        and not st.session_state.analyzed:
    st.session_state.page = "Resume Upload"

PAGE_FUNCS[st.session_state.page]()

Overwriting app.py


In [7]:
import os
print(os.path.getsize("app.py"))  # should print ~24000

24654
